# S21 · Yes/no decisions — packing a box and matching agents

Today's decisions are switches: put this parcel in the delivery box, or not?
Give this agent that route, or not? We write two small models in PuLP and let a
free solver make the choices for us. Along the way we see why "just round the
easy answer" can go wrong.

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on
  each cell, top to bottom, and read the plain note above each one.
- New to the idea of a goal, some rules, and a solver that finds the best choice?
  Open `primers/optimization_objectives_and_constraints.md` for a ten-minute,
  picture-first version. It is the foundation for today.
- Already confident with code or with optimization? Run the simple cells, then
  look for the ones marked **Stretch (optional)**.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already
installed everything with `uv`, so it does nothing there.

In [ ]:
# The only library Colab does not already ship is PuLP, so we install just that.
import sys
if "google.colab" in sys.modules:
    !pip install -q "pulp<4"
else:
    print("Not on Colab - assuming the libraries are already installed.")

In [ ]:
import pulp                 # writes the model and hands it to a free solver (CBC)
import numpy as np             # fast maths on tables of numbers

## Step 1 — the packing problem in plain words

A delivery rider has a box that can carry a limited **weight**. Each parcel has a
**value** to us (think priority, or the worth of the goods) and a **weight** in
kg. We must choose a set of parcels so the total value is as large as possible,
without going over the weight limit.

For each parcel we make a yes/no decision: take it (1) or leave it (0). That
yes/no is a **binary variable**, the star of today's session. This little puzzle
has a name: the **knapsack problem**.

In [ ]:
# We list the parcels as three plain Python lists, all in the same order.
# (Parcel 0 is the phone, parcel 1 the mixer, and so on.)
parcel_names   = ["phone", "mixer", "books", "medicines", "cookware"]
parcel_values  = [6, 4, 3, 1, 5]      # how much we want to deliver each one
parcel_weights = [2, 1, 3, 1, 4]      # how heavy each parcel is, in kg

# The box can carry at most 6 kg.
weight_limit = 6

# How many parcels there are in total.
number_of_parcels = len(parcel_names)

print("We have", number_of_parcels, "parcels and a box limit of", weight_limit, "kg.")

## Step 2 — create the problem and the yes/no variables

We tell PuLP we are **maximising**, because we want the most value. Then we make
one binary variable per parcel: `take_phone`, `take_mixer`, and so on. Each can
only be 0 (leave it) or 1 (take it). That is exactly what `cat="Binary"` means.

In [ ]:
# Create the problem. LpMaximize means "make the goal as big as possible".
knapsack = pulp.LpProblem("knapsack", pulp.LpMaximize)

# One yes/no variable per parcel. We keep them in a list so we can loop over them.
take_parcel = []
for parcel_index in range(number_of_parcels):
    variable_name = "take_" + parcel_names[parcel_index]
    variable = pulp.LpVariable(variable_name, cat="Binary")
    take_parcel.append(variable)

print("Created", len(take_parcel), "binary (0/1) decision variables.")

## Step 3 — write the goal and the rule

The **goal** (the objective) is the total value of the parcels we take: for each
parcel, its value if we take it, zero if we don't. The **rule** (the constraint)
is that their total weight must not exceed the box limit.

We build each sum the plain way: make a list of the terms with a `for` loop, then
add the list up with `pulp.lpSum`.

In [ ]:
# Goal: total value of the parcels we decide to take.
# For each parcel, value * take is its value when take=1, or 0 when take=0.
value_terms = []
for parcel_index in range(number_of_parcels):
    value_terms.append(parcel_values[parcel_index] * take_parcel[parcel_index])
total_value = pulp.lpSum(value_terms)
knapsack += total_value

# Rule: the total weight of the parcels we take must be at most the limit.
weight_terms = []
for parcel_index in range(number_of_parcels):
    weight_terms.append(parcel_weights[parcel_index] * take_parcel[parcel_index])
total_weight = pulp.lpSum(weight_terms)
knapsack += total_weight <= weight_limit

print("Goal and weight rule added to the model.")

## Step 4 — let the solver choose

We call the free CBC solver that comes with PuLP. `msg=0` just hides pages of
internal log lines, so we see only our own results.

In [ ]:
# Solve the problem with the free CBC solver.
knapsack.solve(pulp.PULP_CBC_CMD(msg=0))

# PuLP tells us whether it found the best possible answer.
print("Solver status:", pulp.LpStatus[knapsack.status])

## Step 5 — read off the answer

We look at each variable. If the solver set it to 1, that parcel goes in the box.
We also print the total value and total weight to check we stayed under the
limit.

In [ ]:
chosen_value = 0
chosen_weight = 0

print("Parcels the solver chose to load:")
for parcel_index in range(number_of_parcels):
    # value() reads back the number the solver chose for this variable (0 or 1).
    if pulp.value(take_parcel[parcel_index]) == 1:
        print("  -", parcel_names[parcel_index],
              "(value", parcel_values[parcel_index],
              ", weight", parcel_weights[parcel_index], "kg)")
        chosen_value = chosen_value + parcel_values[parcel_index]
        chosen_weight = chosen_weight + parcel_weights[parcel_index]

print()
print("Total value :", chosen_value)
print("Total weight:", chosen_weight, "kg  (limit was", weight_limit, "kg)")

### Stretch (optional) — check the solver by trying every combination

Skip this if you are new to code. With only five parcels there are just 2⁵ = 32
possible yes/no combinations, so we can quietly try them all and confirm the
solver really did find the best. (This brute-force check is only possible because
the problem is tiny. With 50 parcels there would be over a thousand trillion
combinations, which is the whole reason we need a solver.)

In [ ]:
import itertools

best_value_found = 0
best_combo = None

# Each combo is a tuple of 0/1 choices, one per parcel: e.g. (1, 0, 1, 0, 1).
for combo in itertools.product([0, 1], repeat=number_of_parcels):
    total_w = sum(combo[i] * parcel_weights[i] for i in range(number_of_parcels))
    total_v = sum(combo[i] * parcel_values[i] for i in range(number_of_parcels))
    if total_w <= weight_limit and total_v > best_value_found:
        best_value_found = total_v
        best_combo = combo

print("Best value by brute force:", best_value_found)
print("Best value from the solver:", chosen_value)
print("They match - the solver found the true best.")

## Step 6 — why not just drop the yes/no rule and round?

It is tempting to ignore the 0/1 rule, let each `take` be any fraction between 0
and 1, and round afterwards. Solving the problem **without** the whole-number rule
is called the **LP relaxation**. Let's try it on a tiny case to see why rounding
is not safe.

Tiny case: one parcel worth 10, weighing 3 kg, and a box that holds only 2 kg.
The honest yes/no answer is *leave it* (it does not fit). But the relaxed version
is allowed to take a *fraction* of the parcel.

In [ ]:
# Same kind of model, but the variable can be any fraction from 0 to 1
# instead of strictly 0 or 1. This is the "LP relaxation".
relaxed = pulp.LpProblem("relaxed", pulp.LpMaximize)

# lowBound=0, upBound=1 lets the variable slide between 0 and 1.
fraction_taken = pulp.LpVariable("fraction_taken", lowBound=0, upBound=1)

relaxed += 10 * fraction_taken                 # goal: as much value as possible
relaxed += 3 * fraction_taken <= 2             # rule: weight must fit in 2 kg

relaxed.solve(pulp.PULP_CBC_CMD(msg=0))

print("Relaxed answer (allowed to be a fraction):", round(pulp.value(fraction_taken), 3))
print("That says 'pack two-thirds of the parcel' - which makes no real sense.")
print("The honest 0/1 answer is 0: the parcel simply does not fit.")

So the relaxed problem says **0.667** of the parcel, but you cannot pack
two-thirds of a box of cookware. Round that down and you get 0; round it up and
you get 1, which breaks the weight limit. This is why we let the solver enforce
whole numbers for us, instead of relaxing and rounding by hand. This is the one
idea of the whole session.

## Step 7 — a second model: matching agents to routes

Now a different yes/no decision, the **assignment problem**. We have the same
number of delivery **agents** and **routes**, and a cost (say, hours) for each
agent doing each route. We want to give every agent exactly one route and every
route exactly one agent, so that the **total cost is as small as possible**.

In [ ]:
# Set a seed so the made-up costs are the same every time we run this.
np.random.seed(0)

agent_names = ["Asha", "Bilal", "Chitra", "Dev"]
route_names = ["Route A", "Route B", "Route C", "Route D"]

number_of_agents = len(agent_names)
number_of_routes = len(route_names)

# A cost table: cost[i][j] is the hours agent i takes on route j.
# We make up small whole-number costs between 2 and 9.
cost = np.random.randint(2, 10, size=(number_of_agents, number_of_routes))

print("Cost table (rows = agents, columns = routes):")
print(cost)

## Step 8 — a yes/no variable for every (agent, route) pair

The decision `assign[i][j]` is 1 if agent `i` is given route `j`, and 0
otherwise. With 4 agents and 4 routes that is 16 binary variables. We store them
in a list of lists so we can reach `assign[i][j]`.

In [ ]:
# This time we MINIMISE the total cost.
assignment = pulp.LpProblem("assignment", pulp.LpMinimize)

# Build a grid of binary variables: assign[i][j].
assign = []
for agent_index in range(number_of_agents):
    row = []
    for route_index in range(number_of_routes):
        variable_name = "assign_" + str(agent_index) + "_" + str(route_index)
        variable = pulp.LpVariable(variable_name, cat="Binary")
        row.append(variable)
    assign.append(row)

print("Created a", number_of_agents, "x", number_of_routes, "grid of 0/1 variables.")

## Step 9 — the goal and the two matching rules

**Goal:** add up the cost of every assignment we switch on.

**Rule 1:** each agent gets exactly one route, so the variables in that agent's
row must sum to 1.

**Rule 2:** each route gets exactly one agent, so the variables in that route's
column must sum to 1.

In [ ]:
# Goal: total cost over all the (agent, route) pairs we switch on.
cost_terms = []
for agent_index in range(number_of_agents):
    for route_index in range(number_of_routes):
        this_cost = int(cost[agent_index][route_index])
        cost_terms.append(this_cost * assign[agent_index][route_index])
assignment += pulp.lpSum(cost_terms)

# Rule 1: every agent gets exactly one route (each row sums to 1).
for agent_index in range(number_of_agents):
    row_terms = []
    for route_index in range(number_of_routes):
        row_terms.append(assign[agent_index][route_index])
    assignment += pulp.lpSum(row_terms) == 1

# Rule 2: every route gets exactly one agent (each column sums to 1).
for route_index in range(number_of_routes):
    column_terms = []
    for agent_index in range(number_of_agents):
        column_terms.append(assign[agent_index][route_index])
    assignment += pulp.lpSum(column_terms) == 1

print("Goal and the two matching rules added.")

## Step 10 — solve and print the matching

We solve, then for each agent we find the one route whose variable came back
as 1.

In [ ]:
assignment.solve(pulp.PULP_CBC_CMD(msg=0))
print("Solver status:", pulp.LpStatus[assignment.status])
print()

total = 0
for agent_index in range(number_of_agents):
    for route_index in range(number_of_routes):
        if pulp.value(assign[agent_index][route_index]) == 1:
            this_cost = int(cost[agent_index][route_index])
            total = total + this_cost
            print(agent_names[agent_index], "->", route_names[route_index],
                  " (cost", this_cost, ")")

print()
print("Total cost of this assignment:", total, "hours")

## What you just did

You wrote two integer models in PuLP using only binary 0/1 variables. The
**knapsack** picked parcels to maximise value under a weight limit; the
**assignment** problem matched agents to routes at least cost. You also saw why
dropping the whole-number rule and rounding can give an answer that makes no real
sense, which is exactly why we let the solver enforce whole numbers.

Next notebook: `02_facility_location.ipynb`, where we decide which dark stores a
quick-commerce company should open across a metro.